### PART 1 — Data Collection & Preprocessing

#### We will:

#### - Query ChEMBL for COX-2 (PTGS2)

#### - Retrieve bioactivity data (IC50)

#### - Clean and standardize

#### - Convert IC50 → pIC50

#### - Save clean dataset

In [25]:
# importing the libraries
import os
import pandas as pd
import numpy as np
from chembl_webresource_client.new_client import new_client

In [3]:
# Searching for COX-2 Target in ChEMBL

target = new_client.target
target_query = target.search('cyclooxygenase-2')
targets = pd.DataFrame(target_query)

targets.head()


,cross_references,organism,pref_name,score,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Canis lupus familiaris,Cyclooxygenase-2,20.0,False,CHEMBL4033,"[{'accession': 'Q8SPQ9', 'component_descriptio...",SINGLE PROTEIN,9615.0
1,[],Felis catus,Cyclooxygenase 2,20.0,False,CHEMBL5303558,"[{'accession': 'A8QIU4', 'component_descriptio...",SINGLE PROTEIN,9685.0
2,[],Equus caballus,Cyclooxygenase-1,19.0,False,CHEMBL5303565,"[{'accession': 'Q9N2C8', 'component_descriptio...",SINGLE PROTEIN,9796.0
3,[],Homo sapiens,Cyclooxygenase,17.0,False,CHEMBL2094253,"[{'accession': 'P35354', 'component_descriptio...",PROTEIN FAMILY,9606.0
4,[],Rattus norvegicus,Cyclooxygenase,17.0,False,CHEMBL2095157,"[{'accession': 'P35355', 'component_descriptio...",PROTEIN FAMILY,10116.0


In [4]:
# filtring for humain

targets[targets['organism'] == 'Homo sapiens']


,cross_references,organism,pref_name,score,species_group_flag,target_chembl_id,target_components,target_type,tax_id
3,[],Homo sapiens,Cyclooxygenase,17.0,False,CHEMBL2094253,"[{'accession': 'P35354', 'component_descriptio...",PROTEIN FAMILY,9606.0
6,[],Homo sapiens,Thromboxane-A synthase,15.0,False,CHEMBL1835,"[{'accession': 'P24557', 'component_descriptio...",SINGLE PROTEIN,9606.0
16,[],Homo sapiens,COX-1/COX-2,14.0,False,CHEMBL4523964,"[{'accession': 'P35354', 'component_descriptio...",SELECTIVITY GROUP,9606.0
18,[],Homo sapiens,Prostaglandin G/H synthase 2,13.0,False,CHEMBL230,"[{'accession': 'P35354', 'component_descriptio...",SINGLE PROTEIN,9606.0
22,[],Homo sapiens,Prostaglandin G/H synthase 1,9.0,False,CHEMBL221,"[{'accession': 'P23219', 'component_descriptio...",SINGLE PROTEIN,9606.0
...,...,...,...,...,...,...,...,...,...
3356,[],Homo sapiens,Sodium channel alpha subunit,0.0,False,CHEMBL2331043,"[{'accession': 'P35498', 'component_descriptio...",PROTEIN FAMILY,9606.0
3358,[],Homo sapiens,Mitochondrial complex I (NADH dehydrogenase),0.0,False,CHEMBL2363065,"[{'accession': 'P03923', 'component_descriptio...",PROTEIN COMPLEX,9606.0
3359,[],Homo sapiens,"3',5'-cyclic phosphodiesterase",0.0,False,CHEMBL2363066,"[{'accession': 'O76074', 'component_descriptio...",PROTEIN FAMILY,9606.0
3368,[],Homo sapiens,PI3-kinase class I,0.0,False,CHEMBL3559703,"[{'accession': 'P27986', 'component_descriptio...",PROTEIN COMPLEX GROUP,9606.0


In [5]:
target_id = "CHEMBL230"  

In [6]:
# Retrive Bioactivity Data(IC50 Only)

activity = new_client.activity

res = activity.filter(
    target_chembl_id=target_id
).filter(
    standard_type="IC50"
)

df = pd.DataFrame(res)
df.head()

,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,None,34205,[],CHEMBL762912,In vitro inhibitory activity against human pro...,B,None,None,BAO_0000190,...,Homo sapiens,Prostaglandin G/H synthase 2,9606,None,None,IC50,uM,UO_0000065,None,0.06
1,None,None,34209,[],CHEMBL762912,In vitro inhibitory activity against human pro...,B,None,None,BAO_0000190,...,Homo sapiens,Prostaglandin G/H synthase 2,9606,None,None,IC50,uM,UO_0000065,None,3.23
2,None,None,35476,[],CHEMBL762912,In vitro inhibitory activity against human pro...,B,None,None,BAO_0000190,...,Homo sapiens,Prostaglandin G/H synthase 2,9606,None,None,IC50,uM,UO_0000065,None,0.08
3,None,None,36218,[],CHEMBL769655,Tested in vitro for inhibition against Prostag...,B,None,None,BAO_0000190,...,Homo sapiens,Prostaglandin G/H synthase 2,9606,None,None,IC50,nM,UO_0000065,None,0.12
4,None,None,36708,[],CHEMBL762912,In vitro inhibitory activity against human pro...,B,None,None,BAO_0000190,...,Homo sapiens,Prostaglandin G/H synthase 2,9606,None,None,IC50,uM,UO_0000065,None,100.0


In [7]:
# Keeping the relevant columns 

df = df[['molecule_chembl_id',
         'canonical_smiles',
         'standard_value',
         'standard_units']]

df.head()

,molecule_chembl_id,canonical_smiles,standard_value,standard_units
0,CHEMBL297008,Cc1ccc(-c2ccc(S(C)(=O)=O)cc2)n1-c1ccccc1,60.0,nM
1,CHEMBL289813,Cc1c(C=O)cc(-c2ccc(S(C)(=O)=O)cc2)n1-c1ccc(F)cc1,3230.0,nM
2,CHEMBL43736,Cc1c(COc2cccc(Cl)c2)cc(-c2ccc(S(C)(=O)=O)cc2)n...,80.0,nM
3,CHEMBL140167,Fc1ccc(-c2[nH]c(-c3ccc(F)cc3)c3c2C2CCC3CC2)cc1,0.12,nM
4,CHEMBL44194,CCc1ccc(-c2ccc(S(C)(=O)=O)cc2)n1-c1ccc(F)cc1,100000.0,nM


### Data Cleaning


In [8]:
# Removing missing values

df = df.dropna()

In [9]:
# Keeping only the nM values

df = df[df['standard_units'] == 'nM']

In [10]:
# Convert IC50 to numeric

df['standard_value'] = pd.to_numeric(df['standard_value'])

In [11]:
# Removing duplicates

df = df.drop_duplicates(subset='molecule_chembl_id')

In [13]:
# Converting IC50 → pIC50

df['pIC50'] = -np.log10(df['standard_value'] * 1e-9)

df[['standard_value', 'pIC50']].head()

C:\Users\ElMehdi\anaconda3\envs\cox2_ml\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


,standard_value,pIC50
0,60.00,7.221849
1,3230.00,5.490797
2,80.00,7.096910
3,0.12,9.920819
4,100000.00,4.000000


In [15]:
### Quality check

In [17]:
# Dataset size

print("Number of compounds:", df.shape[0])

Number of compounds: 5377


In [27]:
# Saving the clean dataset

df.to_csv("cox2_clean.csv", index=False)

print("Fichier sauvegardé au même endroit que ce notebook.")


Fichier sauvegardé au même endroit que ce notebook.
